In [ ]:
from pathlib import Path

BASE_DIR = Path(".")

CONFIGS = [
    {"cache_dir": "cache_calibration_propaganda950",        "dataset": "English",    "model": "Qwen 4B"},
    {"cache_dir": "cache_calibration_qwen8b_propaganda950", "dataset": "English",    "model": "Qwen 8B"},
    {"cache_dir": "cache_calibration_translated",           "dataset": "Translated", "model": "Qwen 4B"},
    {"cache_dir": "cache_calibration_qwen8b_translated",    "dataset": "Translated", "model": "Qwen 8B"},
    {"cache_dir": "cache_calibration_ukrainian",            "dataset": "Ukrainian",  "model": "Qwen 4B"},
    {"cache_dir": "cache_calibration_qwen8b_ukrainian",     "dataset": "Ukrainian",  "model": "Qwen 8B"},
]

MODES = ["text", "image", "image+text"]
N_BINS = 10


In [ ]:
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sys.path.insert(0, str(BASE_DIR.resolve()))
from calibration_metrics import multilabel_calibration_report


In [ ]:
UNIQUE_LABELS = [
    "Appeal to (Strong) Emotions",
    "Appeal to authority",
    "Appeal to fear/prejudice",
    "Bandwagon",
    "Black-and-white Fallacy/Dictatorship",
    "Causal Oversimplification",
    "Doubt",
    "Exaggeration/Minimisation",
    "Flag-waving",
    "Glittering generalities (Virtue)",
    "Loaded Language",
    "Misrepresentation of Someone's Position (Straw Man)",
    "Name calling/Labeling",
    "Obfuscation, Intentional vagueness, Confusion",
    "Presenting Irrelevant Data (Red Herring)",
    "Reductio ad hitlerum",
    "Repetition",
    "Slogans",
    "Smears",
    "Thought-terminating cliché",
    "Transfer",
    "Whataboutism",
    "NO_PROPAGANDA",
]


In [ ]:
def load_records(cache_dir, mode):
    mode_subdir = mode.replace("+", "_")
    manifest = json.loads((BASE_DIR / cache_dir / mode_subdir / "manifest.json").read_text())
    records = []
    for entry in manifest:
        p = Path(entry["path"])
        if not p.is_absolute():
            p = BASE_DIR / p
        records.append(json.loads(p.read_text()))
    return records


def clean_labels(labels):
    s = set(labels) if labels else set()
    return {"NO_PROPAGANDA"} if "NO_PROPAGANDA" in s else s


def build_arrays(records):
    per_conf = {lbl: [] for lbl in UNIQUE_LABELS}
    per_corr = {lbl: [] for lbl in UNIQUE_LABELS}

    for r in records:
        gold = clean_labels(r.get("gold_labels", []))
        conf_dict = r.get("confidence") or {}
        for label in UNIQUE_LABELS:
            conf_val = conf_dict.get(label, 0.0)
            try:
                conf_val = float(conf_val)
                if not (0.0 <= conf_val <= 1.0):
                    conf_val = 0.0
            except (TypeError, ValueError):
                conf_val = 0.0
            per_conf[label].append(conf_val)
            per_corr[label].append(1.0 if label in gold else 0.0)

    return {lbl: (np.array(per_conf[lbl]), np.array(per_corr[lbl])) for lbl in UNIQUE_LABELS}


In [ ]:
all_reports = {}
for cfg in CONFIGS:
    for mode in MODES:
        key = (cfg["dataset"], cfg["model"], mode)
        arrays = build_arrays(load_records(cfg["cache_dir"], mode))
        class_conf = {lbl: arrays[lbl][0] for lbl in UNIQUE_LABELS}
        class_corr = {lbl: arrays[lbl][1] for lbl in UNIQUE_LABELS}
        all_reports[key] = multilabel_calibration_report(class_conf, class_corr, n_bins=N_BINS)


In [ ]:
rows = []
for cfg in CONFIGS:
    for mode in MODES:
        rep = all_reports[(cfg["dataset"], cfg["model"], mode)]
        rows.append({
            "dataset":     cfg["dataset"],
            "model":       cfg["model"],
            "mode":        mode,
            "macro_ece":   rep.macro_ece,
            "micro_ece":   rep.micro_ece,
            "macro_ace":   rep.macro_ace,
            "macro_brier": rep.macro_brier,
            "micro_brier": rep.micro_brier,
        })

df = pd.DataFrame(rows)
df.round(4)


## Q2 — Cross-Dataset Calibration

In [ ]:
DATASET_ORDER = ["English", "Translated", "Ukrainian"]
MODE_PALETTE  = {"text": "#4e79a7", "image": "#f28e2b", "image+text": "#59a14f"}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, ylabel in zip(axes, ["macro_ece", "macro_brier"], ["Macro ECE", "Macro Brier"]):
    sub = df.groupby(["dataset", "mode"])[metric].mean().reset_index()
    sub["dataset"] = pd.Categorical(sub["dataset"], categories=DATASET_ORDER, ordered=True)
    sub = sub.sort_values("dataset")

    x = np.arange(len(DATASET_ORDER))
    width = 0.25
    for i, mode in enumerate(MODES):
        vals = [sub.loc[(sub.dataset == d) & (sub["mode"] == mode), metric].values[0] for d in DATASET_ORDER]
        ax.bar(x + i * width, vals, width, label=mode, color=MODE_PALETTE[mode], alpha=0.85)

    ax.set_xticks(x + width)
    ax.set_xticklabels(DATASET_ORDER)
    ax.set_ylabel(ylabel)
    ax.set_title(f"{ylabel} by Dataset & Mode (avg over 4B + 8B)")
    ax.legend(title="Mode")
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))

plt.tight_layout()
plt.show()

df.groupby(["dataset", "mode"])[["macro_ece", "macro_brier"]].mean().round(4)


## Q3 — Model Size: Qwen 4B vs 8B

In [ ]:
MODEL_PALETTE = {"Qwen 4B": "#e15759", "Qwen 8B": "#76b7b2"}

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)

for ax, mode in zip(axes, MODES):
    sub = df[df["mode"] == mode].copy()
    sub["dataset"] = pd.Categorical(sub["dataset"], categories=DATASET_ORDER, ordered=True)
    sub = sub.sort_values("dataset")

    x = np.arange(len(DATASET_ORDER))
    width = 0.35
    for i, (model, color) in enumerate(MODEL_PALETTE.items()):
        vals = [sub.loc[(sub.dataset == d) & (sub.model == model), "macro_ece"].values[0] for d in DATASET_ORDER]
        ax.bar(x + i * width, vals, width, label=model, color=color, alpha=0.85)

    ax.set_xticks(x + width / 2)
    ax.set_xticklabels(DATASET_ORDER)
    ax.set_ylabel("Macro ECE")
    ax.set_title(f"Mode: {mode}")
    ax.legend(title="Model")
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))

plt.tight_layout()
plt.show()

delta_rows = []
for mode in MODES:
    for dataset in DATASET_ORDER:
        ece_4b = df.loc[(df.model == "Qwen 4B") & (df.dataset == dataset) & (df["mode"] == mode), "macro_ece"].values[0]
        ece_8b = df.loc[(df.model == "Qwen 8B") & (df.dataset == dataset) & (df["mode"] == mode), "macro_ece"].values[0]
        delta_rows.append({"mode": mode, "dataset": dataset, "delta": round(ece_8b - ece_4b, 4)})

delta_df = pd.DataFrame(delta_rows).pivot(index="dataset", columns="mode", values="delta").loc[DATASET_ORDER, MODES]
delta_df.style.background_gradient(cmap="RdYlGn_r", axis=None).format("{:.4f}")


## Overview Heatmap

In [ ]:
heatmap_data = df.copy()
heatmap_data["config"] = heatmap_data["dataset"] + "\n" + heatmap_data["model"]
CONFIG_ORDER = [f"{d}\n{m}" for d in DATASET_ORDER for m in ["Qwen 4B", "Qwen 8B"]]
pivot_hm = heatmap_data.pivot(index="config", columns="mode", values="macro_ece").loc[CONFIG_ORDER, MODES]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(pivot_hm, annot=True, fmt=".4f", cmap="YlOrRd", linewidths=0.5, ax=ax, cbar_kws={"label": "Macro ECE"})
ax.set_title("Macro ECE — All Configs × Modes")
ax.set_xlabel("Mode")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


## Per-Label Analysis

In [ ]:
label_ece = {dataset: {lbl: [] for lbl in UNIQUE_LABELS} for dataset in DATASET_ORDER}

for cfg in CONFIGS:
    for mode in MODES:
        rep = all_reports[(cfg["dataset"], cfg["model"], mode)]
        for lbl in UNIQUE_LABELS:
            if lbl in rep.per_class:
                label_ece[cfg["dataset"]][lbl].append(rep.per_class[lbl].ece)

label_rows = []
for lbl in UNIQUE_LABELS:
    row = {"label": lbl}
    for dataset in DATASET_ORDER:
        vals = label_ece[dataset][lbl]
        row[dataset] = np.mean(vals) if vals else float("nan")
    label_rows.append(row)

label_df = pd.DataFrame(label_rows).set_index("label")

fig, axes = plt.subplots(1, len(DATASET_ORDER), figsize=(18, 5), sharey=False)
for ax, dataset in zip(axes, DATASET_ORDER):
    col = label_df[dataset].dropna().sort_values(ascending=False)
    combined = pd.concat([col.head(5), col.tail(5).sort_values()])
    colors = ["#e15759"] * 5 + ["#76b7b2"] * 5
    combined.plot(kind="barh", ax=ax, color=colors, alpha=0.85)
    ax.axvline(0, color="black", linewidth=0.5)
    ax.set_title(f"{dataset} (red=worst, teal=best)")
    ax.set_xlabel("Avg ECE")
    ax.tick_params(axis="y", labelsize=8)

plt.tight_layout()
plt.show()

label_df["mean"] = label_df.mean(axis=1)
label_df.sort_values("mean", ascending=False).round(4)


In [ ]:
out = {}
for cfg in CONFIGS:
    key_name = f"{cfg['dataset']}_{cfg['model'].replace(' ', '_')}"
    out[key_name] = {}
    for mode in MODES:
        rep = all_reports[(cfg["dataset"], cfg["model"], mode)]
        out[key_name][mode] = {
            "macro_ece":   rep.macro_ece,
            "micro_ece":   rep.micro_ece,
            "macro_ace":   rep.macro_ace,
            "macro_brier": rep.macro_brier,
            "micro_brier": rep.micro_brier,
            "per_label": {
                lbl: {"ece": r.ece, "brier": r.brier, "overconfidence": r.overconfidence,
                      "mean_confidence": r.mean_confidence, "accuracy": r.accuracy}
                for lbl, r in rep.per_class.items()
            },
        }

(BASE_DIR / "calibration_comparison_results.json").write_text(json.dumps(out, indent=2))
